# Gains & Guide — Colab에서 LoRA 파인튜닝 (LLaMA-Factory)

**Cursor/AI는 Colab 계정에 직접 연결할 수 없습니다.** 이 노트북을 [Google Colab](https://colab.research.google.com)에서 **파일 → 노트북 업로드**로 열고, GPU 런타임을 선택한 뒤 위에서부터 실행하세요.

**사전 준비**
1. 로컬에서 `cd backend_ai && python -m finetune.build_sft_dataset --validate` 로 `finetune/output/gains_coach_sft_sharegpt.json` 생성
2. [Hugging Face 토큰](https://huggingface.co/settings/tokens) — 기본 베이스 모델(Qwen2.5)은 공개라 필수는 아니지만, 로그인해 두면 속도·한도에 유리합니다. **Meta Llama** 를 쓰려면 [모델 페이지](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)에서 접근 승인을 받은 뒤 **그 계정 토큰**으로 `login` 해야 합니다(승인 없으면 `GatedRepoError` / 403).
3. 아래 셀에서 데이터 JSON 업로드 또는 Drive 경로 지정

자세한 설명: 레포 `docs/ollama_finetune.md`

In [ ]:
!nvidia-smi

In [ ]:
# LLaMA-Factory 설치 (Colab: 수 분 소요)
%cd /content
!rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -q -e ".[torch,metrics]"

In [ ]:
# Hugging Face 로그인 (선택이지만 권장 — 게이트 모델·허브 한도에 필요)
from huggingface_hub import login
import os

token = os.environ.get("HF_TOKEN", "")
if not token:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = input("HF_TOKEN 붙여넣기(엔터만 누르면 스킵): ").strip()
if token:
    login(token=token, add_to_git_credential=False)
else:
    print("토큰 없이 진행합니다. 공개 모델(Qwen2.5)은 보통 동작합니다.")

In [ ]:
# 학습 데이터 넣기 — 방법 A: 업로드
from google.colab import files
import json
import os

DATA_DIR = "/content/LLaMA-Factory/data"
os.makedirs(DATA_DIR, exist_ok=True)

print("gains_coach_sft_sharegpt.json 파일을 선택하세요 (로컬에서 build_sft_dataset 로 생성).")
uploaded = files.upload()
for name in uploaded:
    if name.endswith(".json"):
        open(os.path.join(DATA_DIR, "gains_coach_sft_sharegpt.json"), "wb").write(uploaded[name])
        print("saved:", os.path.join(DATA_DIR, "gains_coach_sft_sharegpt.json"))
        break
else:
    raise SystemExit("JSON 파일을 업로드하지 못했습니다.")

In [ ]:
# dataset_info.json 에 gains_coach_sft 등록 (기존 파일과 병합)
import json
import os

info_path = "/content/LLaMA-Factory/data/dataset_info.json"
entry = {
    "gains_coach_sft": {
        "file_name": "gains_coach_sft_sharegpt.json",
        "formatting": "sharegpt",
        "columns": {"messages": "conversations"},
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_tag": "human",
            "assistant_tag": "gpt",
            "system_tag": "system",
        },
    }
}

if os.path.isfile(info_path):
    with open(info_path, "r", encoding="utf-8") as f:
        base = json.load(f)
else:
    base = {}
base.update(entry)
with open(info_path, "w", encoding="utf-8") as f:
    json.dump(base, f, ensure_ascii=False, indent=2)
print("merged dataset_info.json")

In [ ]:
# Colab용 학습 YAML 작성
# - 기본: Qwen2.5-3B-Instruct (HF 게이트 없음) + template qwen — Colab에서 바로 받기 쉬움
# - Llama 3.2 를 쓰려면: HF 모델 페이지에서 접근 승인 + 승인된 계정의 HF_TOKEN 으로 login 후 아래 두 줄만 교체
#     model_name_or_path: meta-llama/Llama-3.2-3B-Instruct
#     template: llama3
# - 무료 GPU는 보통 T4: fp16 권장. A100/L4면 fp16/false, bf16/true 로 바꿔도 됨
yaml_text = r"""
### model
model_name_or_path: Qwen/Qwen2.5-3B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_target: all

### dataset
dataset: gains_coach_sft
template: qwen
cutoff_len: 2048
overwrite_cache: true
preprocessing_num_workers: 2
dataloader_num_workers: 2

### output
output_dir: saves/gains-coach-lora
logging_steps: 5
save_steps: 50
plot_loss: true
overwrite_output_dir: true
report_to: none

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true
bf16: false
ddp_timeout: 180000000
"""
path = "/content/LLaMA-Factory/gains_coach_colab.yaml"
open(path, "w", encoding="utf-8").write(yaml_text.strip() + "\n")
print("wrote", path)

In [ ]:
# 학습 실행 (실패 시 아래에서 stdout/stderr 를 한 번 더 출력)
%cd /content/LLaMA-Factory

import os
import shutil
import subprocess
import sys

LF_ROOT = "/content/LLaMA-Factory"
os.chdir(LF_ROOT)
cfg = "gains_coach_colab.yaml"

def _train_cmd():
    if shutil.which("llamafactory-cli"):
        return ["llamafactory-cli", "train", cfg]
    if shutil.which("llamafactory"):
        return ["llamafactory", "train", cfg]
    # 최신 LLaMA-Factory: 패키지 엔트리는 llamafactory.cli:main (구 llamafactory.cli.train 아님)
    return [sys.executable, "-m", "llamafactory.cli", "train", cfg]

cmd = _train_cmd()
print("실행:", " ".join(cmd), "(cwd=", LF_ROOT, ")")
# 실시간 로그는 터미널로; 실패 시에만 버퍼 출력으로 원인 확인
p = subprocess.run(cmd, cwd=LF_ROOT)
if p.returncode != 0:
    print("\n=== 학습 프로세스가 비정상 종료했습니다. 마지막 로그 재수집 ===\n")
    r2 = subprocess.run(
        cmd,
        cwd=LF_ROOT,
        capture_output=True,
        text=True,
    )
    if r2.stdout:
        print(r2.stdout[-12000:])
    if r2.stderr:
        print(r2.stderr[-12000:])
    raise subprocess.CalledProcessError(p.returncode, cmd)

## 다음 단계

- 어댑터 머지 후 GGUF 변환은 로컬 또는 RunPod에서 `llama.cpp`로 진행 (`docs/ollama_finetune.md` 4절).
- Colab 세션 끊기면 `/content`는 사라지므로, `saves/gains-coach-lora` 를 **Drive에 복사**하거나 HF Hub에 `push_to_hub` 하세요.

**CLI 오류 시**: 학습 셀은 실패 후 **로그를 다시 출력**합니다. T4에서는 `bf16` 대신 `fp16`, Llama는 HF 라이선스·토큰, OOM은 배치/`cutoff_len` 축소를 확인하세요. [LLaMA-Factory README](https://github.com/hiyouga/LLaMA-Factory)에서 최신 학습 명령도 함께 확인합니다.